In [33]:
import numpy as np

def anisotropic_diffusion_2d(img, niter=10, kappa=30.0, gamma=0.15, option=2):
    """
    Difusão anisotrópica (Perona-Malik) em 2D.
    img:      (H, W) float32 (idealmente normalizada)
    niter:    número de iterações
    kappa:    parâmetro de condutância (controle de sensibilidade a borda)
    gamma:    passo de integração (<= 0.25 para estabilidade em 2D)
    option:   1 -> exp(-(grad/kappa)^2), 2 -> 1 / (1 + (grad/kappa)^2)
    """
    img = img.astype(np.float32)
    h, w = img.shape
    U = img.copy()

    for _ in range(niter):
        # Gradientes nas 4 direções (sem wrap-around)
        nablaN = np.zeros_like(U)
        nablaS = np.zeros_like(U)
        nablaE = np.zeros_like(U)
        nablaW = np.zeros_like(U)

        nablaN[:-1, :] = U[1:, :] - U[:-1, :]
        nablaS[1:,  :] = U[:-1, :] - U[1:,  :]
        nablaE[:, :-1] = U[:, 1:] - U[:, :-1]
        nablaW[:, 1:]  = U[:, :-1] - U[:, 1:]

        if option == 1:
            cN = np.exp(-(nablaN / kappa) ** 2)
            cS = np.exp(-(nablaS / kappa) ** 2)
            cE = np.exp(-(nablaE / kappa) ** 2)
            cW = np.exp(-(nablaW / kappa) ** 2)
        else:  # option 2
            cN = 1.0 / (1.0 + (nablaN / kappa) ** 2)
            cS = 1.0 / (1.0 + (nablaS / kappa) ** 2)
            cE = 1.0 / (1.0 + (nablaE / kappa) ** 2)
            cW = 1.0 / (1.0 + (nablaW / kappa) ** 2)

        U += gamma * (cN * nablaN + cS * nablaS + cE * nablaE + cW * nablaW)

    return U


In [34]:
def anisotropic_diffusion_npy(
    input_npy,
    output_npy=None,
    niter=5,
    kappa=30.0,
    gamma=0.15,
    option=2,
):
    """
    Aplica difusão anisotrópica em todas as imagens de um .npy.

    Suporta:
      - (H, W)
      - (N, H, W)
      - (N, H, W, C)
    """
    arr = np.load(input_npy)
    print("Entrada:", arr.shape, arr.dtype, "min/max:", np.min(arr), np.max(arr))

    arr = arr.astype(np.float32)

    if arr.ndim == 2:
        # Imagem única
        out = anisotropic_diffusion_2d(arr, niter=niter, kappa=kappa,
                                       gamma=gamma, option=option)

    elif arr.ndim == 3:
        # (N, H, W) – várias imagens 2D (ex: sísmica ou slices de volume)
        N, H, W = arr.shape
        out = np.empty_like(arr, dtype=np.float32)
        for i in range(N):
            out[i] = anisotropic_diffusion_2d(arr[i], niter=niter, kappa=kappa,
                                              gamma=gamma, option=option)

    elif arr.ndim == 4:
        # (N, H, W, C) – ex: RGB
        N, H, W, C = arr.shape
        out = np.empty_like(arr, dtype=np.float32)
        for i in range(N):
            for c in range(C):
                out[i, :, :, c] = anisotropic_diffusion_2d(
                    arr[i, :, :, c],
                    niter=niter,
                    kappa=kappa,
                    gamma=gamma,
                    option=option,
                )
    else:
        raise ValueError(f"Shape {arr.shape} não suportado. Use 2D, 3D ou 4D.")

    print("Saída:", out.shape, out.dtype, "min/max:", np.min(out), np.max(out))

    if output_npy is not None:
        np.save(output_npy, out)
        print(f"Salvo em: {output_npy}")

    return out


In [35]:
input_npy  = "rgb_sem_filtro.npy"              # seu .npy de entrada
output_npy = "meu_arquivo_aniso.npy"        # onde salvar o resultado

out = anisotropic_diffusion_npy(
    input_npy,
    output_npy=output_npy,
    niter=5,       # começa com 3–5 pra testar
    kappa=30.0,
    gamma=0.15,
    option=2,
)


Entrada: (1100, 256, 256, 3) float32 min/max: 0.0 1.0
Saída: (1100, 256, 256, 3) float32 min/max: 0.0 1.0
Salvo em: meu_arquivo_aniso.npy
